In [0]:
from pyspark.sql.functions import *

data = [(1, "  alice  ", "123-456-7890", None, "2024-01-15"),
        (2, "BEN", "987.654.3210", "9998887777", "2023-11-02"),
        (3, "Cara", "555 123 4567", None, "2025-03-20")]
columns = ["id", "name", "phone", "mobile", "signup_date"]

df = spark.createDataFrame(data, columns)
df = df.withColumn("signup_date", col("signup_date").cast("date"))
df.display()

In [0]:
# String functions clean up name
df = df.withColumn("name_clean", trim(upper(col("name"))))
df.display()

In [0]:
# Character count of a string
df = df.withColumn("character_count",length(col("name_clean")))
df.display()



In [0]:
from pyspark.sql.functions import regexp_replace

# Extract part of a string
df = df.withColumn("area_code", substring(col("phone"), 1, 3))
df.display()


In [0]:
from pyspark.sql.functions import regexp_replace

# Option 1: Replace literal dots with dashes (escape the dot with \\)
df = df.withColumn("phone_with_dashes", regexp_replace(col("phone"), "\\.", "-"))

# Option 2: Remove all non-digit characters (clean phone numbers)
df = df.withColumn("phone_clean", regexp_replace(col("phone"), "[^0-9]", ""))

# Option 3: Standardize to XXX-XXX-XXXX format
df = df.withColumn("phone_standardized", 
    concat(
        substring(regexp_replace(col("phone"), "[^0-9]", ""), 1, 3),
        lit("-"),
        substring(regexp_replace(col("phone"), "[^0-9]", ""), 4, 3),
        lit("-"),
        substring(regexp_replace(col("phone"), "[^0-9]", ""), 7, 4)
    )
)

df.display()


In [0]:

#Handling Nulls with coalesce() and isnull()
#coalesce() returns the first non-null value from a list of columns — genuinely useful when you have a preferred column and a fallback:

#⚠️ coalesce() is different from isNull()/isNotNull() from Day 3 — isNull() checks whether a value is missing, coalesce() fills in a replacement value when it is. You'll likely use both together: check with isNull(), then fix with coalesce() or fillna().

df = df.withColumn("contact_number", coalesce(col("mobile"), col("phone_clean"), lit("Not Available")))
df.display()

In [0]:
# Date Functions
df.withColumn("signup_year", year(col("signup_date"))).display()
df.withColumn("days_since_signup", datediff(current_date(), col("signup_date"))).display()


In [0]:
# Comparison: coalesce() vs fillna() vs replace()

# Method 1: coalesce() - checks multiple columns
df_coalesce = df.withColumn("contact_method1", 
    coalesce(col("mobile"), col("phone_clean"), lit("Not Available")))

# Method 2: fillna() - only fills nulls in ONE column (can't check multiple sources)
df_fillna = df.withColumn("contact_method2", col("mobile")) \
    .fillna({"contact_method2": "Not Available"})

# Method 3: replace() - replaces specific VALUES (not nulls)
df_replace = df.withColumn("contact_method3", col("mobile")) \
    .replace({"9998887777": "REDACTED"}, subset=["contact_method3"])

print("=== Method 1: coalesce() - Multiple column fallback ===")
df_coalesce.select("id", "mobile", "phone_clean", "contact_method1").display()

print("\n=== Method 2: fillna() - Single column default ===")
df_fillna.select("id", "mobile", "phone_clean", "contact_method2").display()

print("\n=== Method 3: replace() - Replace specific values ===")
df_replace.select("id", "mobile", "contact_method3").display()

In [0]:
#Chaining all together
df_final = (spark.createDataFrame(data, columns)
    .withColumn("signup_date", col("signup_date").cast("date"))
    .withColumn("name_clean", trim(upper(col("name"))))
    .withColumn("phone_clean", regexp_replace(col("phone"), "[^0-9]", ""))
    .withColumn("contact_number", coalesce(col("mobile"), col("phone_clean"), lit("Not Available")))
    .withColumn("days_since_signup", datediff(current_date(), col("signup_date")))
)
df_final.display()